In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
import keras
print(keras.__version__)

In [ ]:
# --------------------------------------------------------------------
# 1. Define your file groups
# --------------------------------------------------------------------
param_set_A_files = (
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-11T14_04_49/caiman").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-12T13_05_34/caiman").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-14T17_19_31/caiman").glob("*.hdf5"))
)

param_set_B_files = list(
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman2").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-11T14_04_49/caiman2").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-12T13_05_34/caiman2").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman2").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-14T17_19_31/caiman2").glob("*.hdf5"))
)

param_set_C_files = list(
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-10T17_53_09/caiman3").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-11T14_04_49/caiman3").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-12T13_05_34/caiman3").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-13T15_32_54/caiman3").glob("*.hdf5")) +
    list(Path("/media/toor/Seagate2/CaimanTesting/2024-06-14T17_19_31/caiman3").glob("*.hdf5"))
)

print("Found A files:", len(param_set_A_files))
for f in param_set_A_files:
    print(" -", f)

print("\nFound B files:", len(param_set_B_files))
for f in param_set_B_files:
    print(" -", f)

print("\nFound C files:", len(param_set_C_files))
for f in param_set_C_files:
    print(" -", f)


In [ ]:

# Function to extract parameters from a CaImAn HDF5 file
def extract_params(file_path):
    params = {}
    with h5py.File(file_path, "r") as f:
        def visitor(name, node):
            if isinstance(node, h5py.Dataset):
                try:
                    value = node[()]
                    if isinstance(value, np.ndarray) and value.shape == ():
                        value = value.item()
                    params[name] = value
                except Exception:
                    params[name] = None
        f['params'].visititems(visitor)
    return pd.Series(params)

# Function to extract number of components from A matrix
def extract_num_components(file_path):
    with h5py.File(file_path, "r") as f:
        shape = f['estimates/A/shape'][()]
        return int(shape[1]) if shape.shape == (2,) else np.nan

# Summarize parameter and component count per file
def summarize_files(file_list, label):
    summary = []
    for file in file_list:
        params = extract_params(file)
        num_comps = extract_num_components(file)
        params['num_components'] = num_comps
        params['file'] = Path(file).name
        params['param_set'] = label
        summary.append(params)
    return pd.DataFrame(summary)


In [ ]:
summary_A = summarize_files(param_set_A_files, 'A')
summary_B = summarize_files(param_set_B_files, 'B')
summary_C = summarize_files(param_set_C_files, 'C')
combined_summary = pd.concat([summary_A, summary_B, summary_C], ignore_index=True)

# Save or preview
combined_summary.to_csv("caiman_param_comparison.csv", index=False)
combined_summary


In [ ]:
param_diff = combined_summary.groupby('param_set').mean(numeric_only=True).T
param_diff['abs_diff'] = np.abs(param_diff['A'] - param_diff['B'])
param_diff.sort_values('abs_diff', ascending=False).head(10)

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.sparse import csc_matrix
from scipy.spatial.distance import cdist

# Load sparse matrix A
def load_sparse_A(h5file):
    with h5py.File(h5file, "r") as f:
        A_data = f['estimates/A/data'][:]
        A_indices = f['estimates/A/indices'][:]
        A_indptr = f['estimates/A/indptr'][:]
        A_shape = f['estimates/A/shape'][:]
        A = csc_matrix((A_data, A_indices, A_indptr), shape=tuple(A_shape))
    return A

# Load C traces
def load_C_traces(h5file):
    with h5py.File(h5file, "r") as f:
        return f['estimates/C'][:]

# Load SNR or compute fallback confidence
def load_snr_or_fallback(C):
    try:
        return np.nanmax(C, axis=1) / np.nanstd(C, axis=1)
    except:
        return np.var(C, axis=1)

# Normalize spatial components for cosine comparison
from sklearn.metrics.pairwise import cosine_similarity

def normalize_spatial(A):
    """Normalize columns of sparse spatial matrix A (pixels x components)."""
    A_dense = A.toarray()  # or .todense() if memory OK
    norms = np.linalg.norm(A_dense, axis=0) + 1e-8  # per component
    A_normed = A_dense / norms
    return A_normed

# Match components from A to B using spatial correlation

def match_components(A1, A2, threshold=0.6, max_components=2000):
    """Compute cosine similarity and find matched/unmatched components."""
    A1_norm = normalize_spatial(A1)
    A2_norm = normalize_spatial(A2)

    # If datasets are very large, subsample to avoid memory explosion
    if A1_norm.shape[1] > max_components or A2_norm.shape[1] > max_components:
        print("⚠️ Large dataset detected, comparing first", max_components, "components per set.")
        A1_norm = A1_norm[:, :max_components]
        A2_norm = A2_norm[:, :max_components]

    # Compute cosine similarity (faster & safer than manual cdist)
    corr = cosine_similarity(A1_norm.T, A2_norm.T)

    maxA1 = corr.max(axis=1)
    matchedA1 = np.where(maxA1 > threshold)[0]
    unmatchedA1 = np.where(maxA1 <= threshold)[0]

    maxA2 = corr.max(axis=0)
    matchedA2 = np.where(maxA2 > threshold)[0]
    unmatchedA2 = np.where(maxA2 <= threshold)[0]

    return matchedA1, unmatchedA1, matchedA2, unmatchedA2, corr

In [ ]:
# Load one file from each parameter set
file_A = param_set_A_files[0]
file_B = param_set_B_files[0]
file_C = param_set_C_files[0]

A_A = load_sparse_A(file_A)
C_A = load_C_traces(file_A)
conf_A = load_snr_or_fallback(C_A)

A_B = load_sparse_A(file_B)
C_B = load_C_traces(file_B)
conf_B = load_snr_or_fallback(C_B)

A_C = load_sparse_A(file_C)
C_C = load_C_traces(file_C)
conf_C = load_snr_or_fallback(C_C)

# Match and get indices
_, unmatched_A, _, unmatched_B, similarity_matrix = match_components(A_A, A_B, threshold=0.9)
_, unmatched_AC, _, unmatched_C, similarity_matrixAC = match_components(A_A, A_C, threshold=0.9)


In [ ]:
# Function to plot temporal traces with confidence
def plot_unmatched_traces(C, conf, unmatched_indices, label, max_plots=10):
    num_plots = min(max_plots, len(unmatched_indices))
    fig, axes = plt.subplots(num_plots, 1, figsize=(12, 2.5 * num_plots), sharex=True)

    if num_plots == 1:
        axes = [axes]  # Make iterable if only one subplot

    for i, idx in enumerate(unmatched_indices[:num_plots]):
        axes[i].plot(C[idx], color='steelblue')
        axes[i].set_title(f"{label}_{idx} (confidence: {conf[idx]:.2f})")
        axes[i].set_ylabel("Activity")
        axes[i].grid(True)

    axes[-1].set_xlabel("Time (frames)")
    plt.suptitle(f"Unmatched Components from {label}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
plot_unmatched_traces(C_A, conf_A, unmatched_A, "A", max_plots=10)
plot_unmatched_traces(C_B, conf_B, unmatched_B, "B", max_plots=10)
plot_unmatched_traces(C_C, conf_C, unmatched_C, "C", max_plots=10)


In [ ]:
def get_matched_pairs(similarity_matrix, threshold=0.6, max_pairs=10):
    matched_pairs = []
    for i, row in enumerate(similarity_matrix):
        j = np.argmax(row)
        if row[j] > threshold:
            matched_pairs.append((i, j))
        if len(matched_pairs) >= max_pairs:
            break
    return matched_pairs

def plot_matched_component_traces(C_A, C_B, matched_pairs, conf_A=None, conf_B=None, label_A="A", label_B="B"):
    num_plots = len(matched_pairs)
    fig, axes = plt.subplots(num_plots, 1, figsize=(12, 2.5 * num_plots), sharex=True)

    if num_plots == 1:
        axes = [axes]

    for i, (idx_A, idx_B) in enumerate(matched_pairs):
        axes[i].plot(C_A[idx_A], label=f"{label_A}_{idx_A} (conf: {conf_A[idx_A]:.2f})" if conf_A is not None else f"{label_A}_{idx_A}", color='blue', alpha = 0.5)
        axes[i].plot(C_B[idx_B], label=f"{label_B}_{idx_B} (conf: {conf_B[idx_B]:.2f})" if conf_B is not None else f"{label_B}_{idx_B}", color='orange', alpha = 0.5)
        axes[i].set_ylabel("Activity")
        axes[i].legend(loc='upper right', fontsize=8)
        axes[i].grid(True)

    axes[-1].set_xlabel("Time (frames)")
    plt.suptitle("Matched Component Traces", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()



In [ ]:
# Find top N matched pairs
matched_pairs = get_matched_pairs(similarity_matrixAC, threshold=0.9, max_pairs=10)

# Plot temporal traces side-by-side
plot_matched_component_traces(C_A, C_C, matched_pairs, conf_A, conf_C, label_A="A", label_B="B")


In [ ]:
import seaborn as sns
sns.heatmap(similarity_matrix, cmap="viridis")
plt.title("Component Spatial Similarity (A vs B)")
plt.xlabel("B Components")
plt.ylabel("A Components")
plt.show()


In [ ]:
# Get max similarity of each component in B to any component in A
max_similarity_B = similarity_matrix.max(axis=0)

# Set threshold for what counts as "matched"
threshold = 0.6

# Boolean check: is every B component matched to an A component?
all_B_found_in_A = np.all(max_similarity_B > threshold)

print("✅ All components in B found in A above threshold:", all_B_found_in_A)

# Optionally list which B components were NOT matched:
unmatched_B_ids = np.where(max_similarity_B <= threshold)[0]
if len(unmatched_B_ids) > 0:
    print(f"❌ {len(unmatched_B_ids)} components in B were NOT matched:")
    print(unmatched_B_ids)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.heatmap(similarity_matrix, cmap='viridis')
plt.title("Spatial Component Similarity: A vs B")
plt.xlabel("Components in B")
plt.ylabel("Components in A")
plt.show()
